In [1]:
import os
import pandas as pd
import numpy as np
import pydicom
import cv2
import torch
import torch.nn as nn
from transformers import Dinov2Model

data_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
dinov2_path = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
test = pd.read_csv(f"{data_dir}/test.csv")
test_series = pd.read_csv(f"{data_dir}/test_series.csv")

finding_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 
                    'Contusion', 'Fracture']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Test studies: {len(test)}")
print(f"Device: {device}")

Test studies: 3
Device: cuda


In [2]:
# Preprocessing

SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1)]

def select_series_5slots(study_id, series_df):
    study_series = series_df[series_df['StudyInstanceUID'] == study_id]
    slots = {}
    for plane, fluid_sensitive in SLOTS:
        key = f"{plane}_{fluid_sensitive}"
        candidates = study_series[
            (study_series['Anatomical_Plane'] == plane) & 
            (study_series['Fluid_Sensitive'] == fluid_sensitive)
        ]
        slots[key] = candidates.iloc[0]['SeriesInstanceUID'] if len(candidates) > 0 else None
    return slots

def select_slice_indices(total_slices, n_slice=16, band=(0.12, 0.88)):
    start = int(total_slices * band[0])
    end = int(total_slices * band[1])
    indices = np.linspace(start, end-1, n_slice).astype(int)
    return indices

def crop_by_mm(pixel_array, pixel_spacing, crop_mm=130.0):
    row_spacing, col_spacing = pixel_spacing
    crop_h = int(crop_mm / row_spacing)
    crop_w = int(crop_mm / col_spacing)
    h, w = pixel_array.shape
    center_h, center_w = h // 2, w // 2
    half_h, half_w = crop_h // 2, crop_w // 2
    top = max(0, center_h - half_h)
    bottom = min(h, center_h + half_h)
    left = max(0, center_w - half_w)
    right = min(w, center_w + half_w)
    return pixel_array[top:bottom, left:right]

def crop_and_resize(pixel_array, pixel_spacing, crop_mm=130.0, final_size=224):
    cropped = crop_by_mm(pixel_array, pixel_spacing, crop_mm)
    resized = cv2.resize(cropped, (final_size, final_size), interpolation=cv2.INTER_LINEAR)
    return resized

print("Preprocessing functions ready")

Preprocessing functions ready


In [3]:
# Model Classes

class SliceAttentionPool(nn.Module):
    def __init__(self, dim=384):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return (x * weights).sum(dim=1)

class DINOv2KneeModel(nn.Module):
    def __init__(self, dinov2_path, num_findings=12, n_slots=5, n_slice=16):
        super().__init__()
        self.dinov2 = Dinov2Model.from_pretrained(dinov2_path)
        self.dim = self.dinov2.config.hidden_size
        self.n_slots = n_slots
        self.n_slice = n_slice
        self.slice_pool = SliceAttentionPool(self.dim)
        self.slot_attn = nn.Sequential(nn.Linear(self.dim, 128), nn.Tanh(), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Linear(self.dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_findings))
    
    def forward(self, slots, mask):
        batch_size = slots.shape[0]
        x = slots.view(batch_size * self.n_slots * self.n_slice, 1, 224, 224).repeat(1, 3, 1, 1)
        features = self.dinov2(pixel_values=x).pooler_output
        features = features.view(batch_size, self.n_slots, self.n_slice, self.dim)
        
        slot_features = torch.stack([self.slice_pool(features[:, s, :, :]) for s in range(self.n_slots)], dim=1)
        
        slot_weights = self.slot_attn(slot_features).squeeze(-1)
        slot_weights = slot_weights.masked_fill(mask == 0, float('-inf'))
        slot_weights = torch.softmax(slot_weights, dim=1).unsqueeze(-1)
        
        final_feature = (slot_features * slot_weights).sum(dim=1)
        return self.head(final_feature)

print("Model class ready")

Model class ready


In [4]:
checkpoint_dir = "/kaggle/input/datasets/mayankptdr/dinov2-final-checkpoints/checkpoints_dino"
print(os.listdir(checkpoint_dir))

['fold0_latest.pt', 'fold0_best.pt', 'fold2_latest.pt', 'fold4_latest.pt', 'fold4_best.pt', 'fold2_best.pt', 'fold1_latest.pt', 'fold1_best.pt', 'fold3_latest.pt', 'fold3_best.pt']


In [5]:
models = []
for fold_num in range(5):
    m = DINOv2KneeModel(dinov2_path, num_findings=12).to(device)
    ckpt = torch.load(f"{checkpoint_dir}/fold{fold_num}_best.pt", map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    models.append(m)
    print(f"Loaded fold{fold_num}, AUROC={ckpt['auroc']:.4f}")

print(f"\nTotal models loaded: {len(models)}")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold0, AUROC=0.7756


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold1, AUROC=0.7955


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold2, AUROC=0.7930


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold3, AUROC=0.7895


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded fold4, AUROC=0.7838

Total models loaded: 5


In [6]:
def load_study_slots_test(study_id):
    slot_map = select_series_5slots(study_id, test_series)
    
    slot_volumes = []
    slot_masks = []
    
    for plane, fs in SLOTS:
        key = f"{plane}_{fs}"
        series_id = slot_map[key]
        
        if series_id is None:
            vol = np.zeros((16, 224, 224), dtype=np.float32)
            mask = 0.0
        else:
            folder = f"{data_dir}/test_series/{study_id}/{series_id}"
            dcm_files = os.listdir(folder)
            slices = [pydicom.dcmread(f"{folder}/{f}") for f in dcm_files]
            slices_sorted = sorted(slices, key=lambda d: int(d.InstanceNumber))
            total_slices = len(slices_sorted)
            indices = select_slice_indices(total_slices, 16)
            pixel_spacing = [float(x) for x in slices_sorted[0].PixelSpacing]
            
            processed = []
            for idx in indices:
                arr = slices_sorted[idx].pixel_array.astype(np.float32)
                arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
                cr = crop_and_resize(arr, pixel_spacing, 130.0, 224)
                processed.append(cr)
            vol = np.stack(processed).astype(np.float32)
            mask = 1.0
        
        slot_volumes.append(vol)
        slot_masks.append(mask)
    
    slots_array = np.stack(slot_volumes)
    mask_array = np.array(slot_masks, dtype=np.float32)
    
    return slots_array, mask_array

print("Function ready")

Function ready


In [7]:
def predict_study_ensemble(study_id):
    slots_array, mask_array = load_study_slots_test(study_id)
    
    slots = torch.tensor(slots_array, dtype=torch.float32).unsqueeze(0).to(device)
    mask = torch.tensor(mask_array, dtype=torch.float32).unsqueeze(0).to(device)
    
    all_probs = []
    with torch.no_grad():
        for m in models:
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                out = torch.sigmoid(m(slots, mask))
            all_probs.append(out.float().cpu().numpy())
    
    avg_probs = np.mean(all_probs, axis=0)[0]
    return avg_probs

print("Ensemble function ready")

Ensemble function ready


In [8]:
predictions = []
for study_id in test['StudyInstanceUID']:
    try:
        probs = predict_study_ensemble(study_id)
        predictions.append([study_id] + list(probs))
        print(f"Predicted: {study_id}")
    except Exception as e:
        print(f"Error on {study_id}: {e}")
        predictions.append([study_id] + [0.5]*12)

submission = pd.DataFrame(predictions, columns=['StudyInstanceUID'] + finding_columns)
print(submission)

Predicted: 1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191
Predicted: 1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776
Predicted: 1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.193555  0.332422   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.544531  0.492188   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.142676  0.378906   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.508203          0.360547   0.575391    0.566406  0.646484   
1         0.509766          0.476562   0.440625    0.439453  0.508984   
2         0.516406          0.290234   0.490430    0.459766  0.574609   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.373047   0.118994  0.485938   0.245117  0.063745  
1  0.623438   0.432422  0.482031   0.523047  0.320898  
2  0.3

In [9]:
sample_sub = pd.read_csv(f"{data_dir}/sample_submission.csv")
print(sample_sub.columns.tolist() == submission.columns.tolist())
print(sample_sub.shape, submission.shape)

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Saved!")

True
(3, 13) (3, 13)
Saved!
